# Noise, Decoherence and Transpilation

**Learning objectives**

- Compare ideal and noisy results
- Measure fidelity degradation
- Transpile into a hardware-like basis

## Environment setup



In [ ]:
# Run once in a fresh Colab/Jupyter environment, then restart the kernel if prompted.
%pip install -q qiskit qiskit-aer matplotlib pylatexenc scipy sympy

In [ ]:
from importlib.metadata import version
for package in ["qiskit", "qiskit-aer", "matplotlib", "pylatexenc", "scipy", "sympy"]:
    print(f"{package:15}: {version(package)}")

## Ideal versus depolarizing noise

A Bell experiment illustrates how gate errors create unwanted outcomes.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.visualization import plot_histogram
bell=QuantumCircuit(2,2); bell.h(0); bell.cx(0,1); bell.measure([0,1],[0,1])
ideal=AerSimulator().run(bell,shots=4096).result().get_counts()
noise=NoiseModel(); noise.add_all_qubit_quantum_error(depolarizing_error(0.01,1),["h","x"]); noise.add_all_qubit_quantum_error(depolarizing_error(0.08,2),["cx"])
noisy_sim=AerSimulator(noise_model=noise); tqc=transpile(bell,noisy_sim)
noisy=noisy_sim.run(tqc,shots=4096).result().get_counts()
print("Ideal",ideal,"Noisy",noisy); display(plot_histogram([ideal,noisy],legend=["ideal","noisy"]))

## Transpilation

Transpilation decomposes abstract circuits into a target gate basis and can optimize depth.

In [ ]:
source=QuantumCircuit(3); source.h(0); source.swap(0,2); source.ccx(0,1,2)
for level in range(4):
    out=transpile(source,basis_gates=["rz","sx","x","cx"],optimization_level=level)
    print("level",level,"depth",out.depth(),"ops",out.count_ops())
display(source.draw("mpl")); display(transpile(source,basis_gates=["rz","sx","x","cx"],optimization_level=3).draw("mpl",fold=-1))